In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess, OptimalBinning
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task4')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task4/data/IT.csv')
oot = pl.read_csv('task4/data/OOT.csv')

var_cols = [c for c in it.columns if c.startswith('VAR_')]
cat_cols = [c for c in var_cols if it[c].dtype == pl.String]
num_cols = [c for c in var_cols if c not in cat_cols]
print(f'Vars: {len(var_cols)}, Num: {len(num_cols)}, Cat: {len(cat_cols)}')

it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))

cutoff = it.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it.filter(pl.col('TIME_DT') <= cutoff)
val = it.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')

In [ ]:
X_train_raw = train.select(var_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val_raw = val.select(var_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot_raw = oot.select(var_cols).to_pandas()

for c in cat_cols:
    for df in [X_train_raw, X_val_raw, X_oot_raw]:
        df[c] = df[c].fillna('__missing__')

In [ ]:
X_train_enc = X_train_raw.copy()
X_val_enc = X_val_raw.copy()
X_oot_enc = X_oot_raw.copy()

enc_map = {}
for c in cat_cols:
    le = LabelEncoder()
    all_vals = list(set(
        X_train_enc[c].unique().tolist() + X_val_enc[c].unique().tolist() + X_oot_enc[c].unique().tolist()
    ))
    le.fit(all_vals)
    for df in [X_train_enc, X_val_enc, X_oot_enc]:
        df[c] = le.transform(df[c])
    enc_map[c] = le

bp = BinningProcess(
    variable_names=var_cols,
    categorical_variables=cat_cols,
    min_prebin_size=0.02,
    min_bin_size=0.05,
    max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train_enc.values, y_train)
selected = list(bp.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)}')

summary = bp.summary().sort_values('iv', ascending=False)
print(summary[['name', 'iv', 'dtype']].head(20).to_string())

In [ ]:
X_tr_woe = bp.transform(X_train_enc.values, metric='woe')
X_va_woe = bp.transform(X_val_enc.values, metric='woe')
X_oo_woe = bp.transform(X_oot_enc.values, metric='woe')
print(f'WOE shape: {X_tr_woe.shape}')

best_gini_lr = 0
best_lr = None
best_C = None

for C in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_tr_woe, y_train)
    p = lr.predict_proba(X_va_woe)[:, 1]
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'WOE+LR C={C}: val_gini={g:.4f}')
    if g > best_gini_lr:
        best_gini_lr, best_lr, best_C = g, lr, C

p_lr_val = best_lr.predict_proba(X_va_woe)[:, 1]
print(f'\nBest LR: C={best_C}, val_gini={best_gini_lr:.4f}')

In [ ]:
sel_cat_cols = [c for c in selected if c in cat_cols]
sel_cat_indices_all = [var_cols.index(c) for c in cat_cols]
sel_cat_indices_sel = [selected.index(c) for c in sel_cat_cols]

X_train_lgb_all = X_train_enc.copy()
X_val_lgb_all = X_val_enc.copy()
X_oot_lgb_all = X_oot_enc.copy()

X_train_lgb_sel = X_train_enc[selected].copy()
X_val_lgb_sel = X_val_enc[selected].copy()
X_oot_lgb_sel = X_oot_enc[selected].copy()

print(f'LGB all: {X_train_lgb_all.shape[1]} features, LGB sel: {X_train_lgb_sel.shape[1]} features')
print(f'Selected cats: {sel_cat_cols}')

In [ ]:
configs = [
    {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 200, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'bagging_freq': 3, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 15, 'learning_rate': 0.03, 'min_child_samples': 300, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 200, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'bagging_freq': 3, 'reg_alpha': 0.5, 'reg_lambda': 0.5},
    {'num_leaves': 7, 'learning_rate': 0.05, 'min_child_samples': 500, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
]

best_lgb_gini = 0
model = None
best_lgb_feat = None

for feat_name, X_tr, X_va, X_oo, cat_idx in [
    ('all', X_train_lgb_all, X_val_lgb_all, X_oot_lgb_all, sel_cat_indices_all),
    ('sel', X_train_lgb_sel, X_val_lgb_sel, X_oot_lgb_sel, sel_cat_indices_sel),
]:
    for i, cfg in enumerate(configs):
        params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'feature_pre_filter': False, **cfg}
        lgb_tr = lgb.Dataset(X_tr, y_train, free_raw_data=False, categorical_feature=cat_idx)
        lgb_va = lgb.Dataset(X_va, y_val, reference=lgb_tr, free_raw_data=False)
        m = lgb.train(params, lgb_tr, num_boost_round=5000, valid_sets=[lgb_va],
                      callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
        p = m.predict(X_va)
        g = 2 * roc_auc_score(y_val, p) - 1
        print(f'{feat_name} cfg{i}: val_gini={g:.4f} (leaves={cfg["num_leaves"]}, lr={cfg["learning_rate"]}, iters={m.best_iteration})')
        if g > best_lgb_gini:
            best_lgb_gini, model, best_lgb_feat = g, m, feat_name

if best_lgb_feat == 'all':
    X_val_lgb, X_oot_lgb = X_val_lgb_all, X_oot_lgb_all
else:
    X_val_lgb, X_oot_lgb = X_val_lgb_sel, X_oot_lgb_sel

p_lgb_val = model.predict(X_val_lgb)
print(f'\nBest LGB ({best_lgb_feat}): val_gini={best_lgb_gini:.4f}')

In [ ]:
best_ens_g = 0
best_w = 0
for w in np.arange(0.0, 1.05, 0.05):
    p_ens = w * p_lr_val + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    if g > best_ens_g:
        best_ens_g, best_w = g, w

print(f'Best ensemble: w_lr={best_w:.2f}, val_gini={best_ens_g:.4f}')

results = {'WOE_LR': best_gini_lr, 'LGB': best_lgb_gini, 'Ensemble': best_ens_g}
best_name = max(results, key=results.get)
val_gini = results[best_name]
print(f'Best: {best_name} = {val_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

In [ ]:
if best_name == 'WOE_LR':
    p_oot_final = best_lr.predict_proba(X_oo_woe)[:, 1]
elif best_name == 'LGB':
    p_oot_final = model.predict(X_oot_lgb)
else:
    p_oot_final = best_w * best_lr.predict_proba(X_oo_woe)[:, 1] + (1 - best_w) * model.predict(X_oot_lgb)

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task4/predictions.csv')
print(f'OOT predictions: {preds.shape}')

with mlflow.start_run(run_name='v5_feature_selection'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('lgb_features', best_lgb_feat)
    mlflow.log_param('n_features_all', len(var_cols))
    mlflow.log_param('n_selected', len(selected))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.log_metric('val_gini_lr', best_gini_lr)
    mlflow.log_metric('val_gini_lgb', best_lgb_gini)
    mlflow.log_metric('val_gini_ens', best_ens_g)
    mlflow.set_tag('task', 'task4')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')